# Swiggy Instamart — Customer Intelligence & Support Copilot

**Capstone walkthrough | Parts A–D | VS Code / Jupyter**

This notebook is a **read-only demonstration** of the existing project outputs, with optional commands to run the original Python scripts. It does not replace or rewrite your working scripts. Run the notebook with your project's `.venv` Python kernel.

**Project structure:** place this notebook in the root folder alongside `README.md`, `data/`, `outputs/`, `part_a/`, `part_b/`, `part_c/`, and `part_d/`.

**Privacy:** do not display or publish unredacted support-ticket text or runtime refund ledgers.

## 0. Setup and locate the project

In [ ]:
from pathlib import Path
import sys
import sqlite3
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "README.md").exists():
    candidates = [Path(r"D:\Capstone Project"), ROOT.parent]
    ROOT = next((p for p in candidates if (p / "README.md").exists()), ROOT)
if not (ROOT / "README.md").exists():
    raise FileNotFoundError("Project README.md not found. Open D:\\Capstone Project in VS Code and select its .venv kernel.")
print("Project:", ROOT)
print("Python:", sys.executable)
print("Pandas:", pd.__version__)


In [ ]:
required = ["data/instamart.db", "outputs/customer_segments.csv", "outputs/lapse_test_predictions.csv", "outputs/redacted_tickets.csv", "outputs/routed_tickets.csv", "outputs/policy_eval_results.csv"]
for name in required:
    print(("✓" if (ROOT / name).exists() else "MISSING"), name)


## Part A — Data cleaning, SQLite, and sales analysis

The original project cleans raw CSVs and builds `data/instamart.db`. The following cells **inspect the saved database** without modifying it.

In [ ]:
db_path = ROOT / "data" / "instamart.db"
with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    display(tables)
    counts = pd.read_sql_query("SELECT status, COUNT(*) AS order_count FROM orders GROUP BY status ORDER BY status", con)
    display(counts)
    revenue = pd.read_sql_query("SELECT ROUND(SUM(order_amount),2) AS completed_revenue_inr FROM orders WHERE status='completed'", con)
    display(revenue)


In [ ]:
with sqlite3.connect(db_path) as con:
    city_category = pd.read_sql_query("""
        SELECT c.city, p.category, ROUND(SUM(o.order_amount), 2) AS revenue_inr
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        ORDER BY c.city, p.category
    """, con)
display(city_category.head(20))
print("City/category total (INR):", round(city_category["revenue_inr"].sum(), 2))


## Part B — Customer segmentation and churn/lapse prediction

**Churn/lapse** means a customer is at risk of stopping orders. Your project uses RFM features, K-Means segments, and machine-learning predictions. The following cells show saved outputs, not retraining.

In [ ]:
segments = pd.read_csv(ROOT / "outputs" / "customer_segments.csv")
print("Customers:", len(segments))
print("Columns:", list(segments.columns))
display(segments.head())
segment_col = next((c for c in segments.columns if c.lower() in ("segment", "segment_label")), None)
if segment_col:
    display(segments[segment_col].value_counts().rename_axis("Segment").reset_index(name="Customers"))
else:
    print("No standard segment-label column found; inspect columns above.")


In [ ]:
pred = pd.read_csv(ROOT / "outputs" / "lapse_test_predictions.csv")
print("Prediction rows:", len(pred))
print("Columns:", list(pred.columns))
display(pred.head(10))
if {"lapsed", "predicted_lapsed"}.issubset(pred.columns):
    from sklearn.metrics import classification_report, confusion_matrix
    print("Confusion matrix:\n",
      confusion_matrix(pred["lapsed"], pred["predicted_lapsed"]))
    print(classification_report(pred["lapsed"], pred["predicted_lapsed"], zero_division=0))
else:
    print("Use the actual/forecast column names shown above to compute classification metrics.")


In [ ]:
retention_path = ROOT / "outputs" / "retention_list.csv"
if retention_path.exists():
    retention = pd.read_csv(retention_path)
    print("Retention candidates:", len(retention))
    display(retention.head(10))


## Part C — Ticket privacy, classification, and routing

Only the **redacted** ticket dataset is previewed here. The routing table summarizes support priorities and queues.

In [ ]:
redacted = pd.read_csv(ROOT / "outputs" / "redacted_tickets.csv")
print("Redacted tickets:", len(redacted))
print("Columns:", list(redacted.columns))
# Show schema/count only by default; avoid inadvertently exposing any remaining identifiers.


In [ ]:
routed = pd.read_csv(ROOT / "outputs" / "routed_tickets.csv")
print("Routed tickets:", len(routed))
print("Columns:", list(routed.columns))
for column in routed.columns:
    if column.lower() in ("priority", "queue", "assigned_queue", "route", "routing_queue"):
        display(routed[column].value_counts(dropna=False).rename_axis(column).reset_index(name="Count"))


## Part D — Policy RAG and refund agent

The project retrieves policy passages, answers questions with the provided **offline `mock_llm.py`**, and implements a refund agent. **Do not run real refund-tool calls in this demonstration**: they may append records to refund ledgers.

In [ ]:
policy_eval = pd.read_csv(ROOT / "outputs" / "policy_eval_results.csv")
print("Policy evaluation rows:", len(policy_eval))
print("Columns:", list(policy_eval.columns))
display(policy_eval.head(10))


In [ ]:
policy_dir = ROOT / "data" / "policies"
print("Policy documents:")
for p in sorted(policy_dir.glob("*.txt")):
    print(" -", p.name)
print("Agent source files:")
for p in sorted((ROOT / "part_d").glob("*.py")):
    print(" -", p.name)


## Optional — Run original scripts from the notebook

The code below is **disabled by default**. Running a script can regenerate outputs, take time, or write files. Run the original scripts in the VS Code terminal as needed, or explicitly uncomment an individual notebook command after reviewing what it does.

```powershell
python part_a/validate_data.py
python part_d/evaluate_policy_rag.py
python part_d/test_agent_limits.py
python part_d/test_resume.py
```

Avoid calling `issue_refund` or running live refund scenarios unless you intend to update the ledger files.

## Submission and viva summary

- **Part A:** Cleaned customer/product/order data, saved SQLite tables, and validated revenue.
- **Part B:** RFM and K-Means segments; churn/lapse prediction using Logistic Regression and Random Forest; retention shortlist.
- **Part C:** PII redaction, ticket classification, token-budget checks, and routing.
- **Part D:** Policy retrieval/RAG, refund eligibility rules, agent tool calls, safeguards, and checkpoints.

**Important:** This notebook is an **explanatory companion** to the full repository. The authoritative implementation remains in the Python scripts. Report metrics from actual notebook outputs and the project README, rather than treating example values as new results.